# SHL 2026 CrisperWhisper Fast Verbatim Transcripts

1. Open this notebook in **Edit** mode. Select **GPU T4 ×2** (one GPU also works),
   turn **Internet on**, set `RUN_TRANSCRIPTION = True` below, then **Run All**.
   The SHL Hiring Assessment 2026 competition is already attached.
2. **No Secrets or API key are needed.** Transcripts are saved directly under
   `/kaggle/working/crisper_dataset` as notebook outputs, like the earlier Whisper run.
3. Transcribe all **769 train + 216 test** WAVs with **CrisperWhisper 2.0 Large**
   in verbatim mode: fillers, repetitions, false starts and original grammar.
   Save word timestamps and raw chunk results. These are ASR predictions;
   exact recovery of every sound is not guaranteed.
4. Training labels stay unchanged. Test labels are **unknown/null**; source test
   CSV labels are placeholders. Join features by **split + filename**.
5. Each recording checkpoints independently. Rerun after interruption to resume.
   To resume in a fresh session, attach an earlier export/dataset containing
   `transcripts_crisper.jsonl`; matching successful records are reused.
6. Save a notebook version **with outputs** after all 985 rows validate.
   These saved notebook outputs can be attached to later notebooks. A private
   transcript dataset can then be created from the saved export separately.
7. The model revision and dependency versions are recorded. Upstream model
   weights **and generated outputs** carry its non-commercial research licence;
   competition data terms also apply. The dataset includes the upstream licence.

[CrisperWhisper documentation](https://github.com/nyrahealth/CrisperWhisper/blob/main/DOCS.md)

8. This separate variant uses the optimized CrisperWhisper CTranslate2 backend with the same pinned Large model and verbatim mode. One-time conversion runs before GPU workers start. The original notebook/run stays intact. Backend changes create a new checkpoint configuration; original Transformers checkpoints are not mixed into this export.


In [ ]:
# Change this to True when YOU are ready to start the Kaggle GPU run.
RUN_TRANSCRIPTION = False

MODEL_ID = "nyralabs/CrisperWhisper2.0_large"
MODEL_REVISION = "f4334f6e8193f2691212d49b20fa12d370e13896"
CRISPER_VERSION = "2.0.3"
BACKEND = "ct2"
COMPUTE_TYPE = "float16"
WORD_TIMESTAMPS = True
HALLUCINATION_MITIGATION = True  # Model's built-in loop repair; no text cleanup.
SEED = 20261012
MAX_ATTEMPTS = 2
DATA_ROOT_OVERRIDE = None  # Normally discovered from the attached competition.

from pathlib import Path
WORK_ROOT = Path("/kaggle/working/crisper_work_ct2")
OUTPUT_DIR = Path("/kaggle/working/crisper_dataset")
MODEL_CACHE = Path("/tmp/crisper_assets_ct2")  # Avoid saving model weights as notebook outputs.
EXPECTED_COUNTS = {"train": 769, "test": 216}
if not RUN_TRANSCRIPTION:
    print("Ready: enable GPU + Internet, set RUN_TRANSCRIPTION=True and Run All. No Secrets needed.")

## 1. Dependencies and GPU checks

1. Internet downloads the pinned package and public model weights.
2. One independent worker is assigned to each available GPU, with recordings
   balanced by duration. Full-recording longform transcription uses the
   model's own 30-second windows and continuation handling.

In [ ]:
if RUN_TRANSCRIPTION:
    import sys, subprocess
    assert sys.version_info >= (3, 11), "Use Kaggle's current Python image (3.11+)."
    import torch
    assert torch.cuda.is_available(), "Select a Kaggle GPU accelerator and restart the session."
    assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 4), "Use a Kaggle image with PyTorch >=2.4."
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
        f"crisperwhisper[ct2]=={CRISPER_VERSION}", "ctranslate2-crisperwhisper==4.7.1.post3", "transformers==4.57.6",
        "accelerate>=0.26,<2", "huggingface_hub>=0.34,<1",
        "soundfile>=0.12", "soxr>=0.3", "pyarrow>=14,<23",
    ])
    GPU_COUNT = torch.cuda.device_count()
    for gpu in range(GPU_COUNT):
        print(f"GPU {gpu}: {torch.cuda.get_device_name(gpu)}")
    print("Dependencies ready; GPU count:", GPU_COUNT)

## 2. Runtime helpers

1. The following cell contains the self-contained checkpoint/export helpers.
2. This notebook does not authenticate with or import the Kaggle API.

In [ ]:
RUNTIME_SOURCE = '"""Kaggle CrisperWhisper transcription, resumable exports and private publication."""\nfrom collections import Counter\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport pandas as pd\nimport soundfile as sf\n\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for block in iter(lambda:handle.read(1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,allow_nan=False).encode()).hexdigest()\n\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_suffix(path.suffix+\'.tmp\')\n    temporary.write_text(json.dumps(value,ensure_ascii=False,allow_nan=False)+\'\\n\')\n    temporary.replace(path)\n\n\ndef make_jobs(data_root,expected_counts=None):\n    """Join audio by split AND filename; preserve source CSV row order and labels."""\n    root=Path(data_root);rows=[];sources={}\n    for split in [\'train\',\'test\']:\n        csv=root/f\'{split}.csv\';table=pd.read_csv(csv)\n        if \'filename\' not in table or table.filename.isna().any() or not table.filename.is_unique:\n            raise ValueError(f\'{split}: source CSV needs unique nonempty filenames\')\n        if split==\'train\' and \'label\' not in table:raise ValueError(\'Missing training labels\')\n        files=list((root/split).rglob(\'*.wav\'))\n        by_name={p.name:p for p in files}\n        if len(by_name)!=len(files) or set(by_name)!=set(table.filename):\n            raise ValueError(f\'{split}: WAV filenames do not match the source CSV exactly\')\n        sources[split]={\'csv_sha256\':digest(csv),\'rows\':len(table)}\n        for source_index,row in table.iterrows():\n            filename=str(row.filename)\n            if Path(filename).name!=filename or not filename.endswith(\'.wav\'):\n                raise ValueError(\'Unexpected audio filename\')\n            label=float(row.label) if split==\'train\' else None\n            if label is not None and (not math.isfinite(label) or not 0<=label<=5):\n                raise ValueError(\'Invalid original training label\')\n            path=by_name[filename];info=sf.info(path)\n            if info.frames<=0:raise ValueError(f\'Empty WAV: {split}/{filename}\')\n            entry=dict(job_index=len(rows),source_row_index=int(source_index),filename=filename,\n                audio_id=filename,split=split,label=label,label_known=split==\'train\',\n                label_source=\'original_train_csv\' if split==\'train\' else \'unlabelled_test\',\n                audio_path=str(path),audio_relative_path=path.relative_to(root).as_posix(),\n                duration_s=float(info.duration),sample_rate_hz=int(info.samplerate),\n                channels=int(info.channels),audio_sha256=digest(path))\n            entry[\'input_fingerprint\']=fingerprint({k:entry[k] for k in [\'split\',\'filename\',\'label\',\n                \'label_known\',\'audio_sha256\',\'duration_s\',\'sample_rate_hz\',\'channels\']})\n            rows.append(entry)\n    counts=dict(Counter(r[\'split\'] for r in rows))\n    if expected_counts is not None and counts!=expected_counts:\n        raise ValueError(f\'Expected {expected_counts}, found {counts}\')\n    return rows,sources\n\n\ndef checkpoint_path(root,row):\n    return Path(root)/row[\'split\']/(row[\'filename\']+\'.json\')\n\n\ndef reusable(record,row,config_hash):\n    return (record.get(\'config_fingerprint\')==config_hash and\n        record.get(\'input_fingerprint\')==row[\'input_fingerprint\'] and\n        record.get(\'split\')==row[\'split\'] and record.get(\'filename\')==row[\'filename\'] and\n        not record.get(\'error\') and isinstance(record.get(\'transcript\'),str) and\n        bool(record[\'transcript\'].strip()))\n\n\ndef read_checkpoint(root,row,config_hash):\n    path=checkpoint_path(root,row)\n    try:record=json.loads(path.read_text())\n    except (OSError,ValueError):return None\n    return record if reusable(record,row,config_hash) else None\n\n\ndef restore_previous(input_root,checkpoint_root,jobs,config_hash):\n    """An attached completed or partial export can resume a new Kaggle session."""\n    by_key={(r[\'split\'],r[\'filename\']):r for r in jobs};restored=0\n    for path in sorted(Path(input_root).rglob(\'transcripts_crisper.jsonl\')):\n        with path.open() as handle:\n            for line in handle:\n                if not line.strip():continue\n                try:record=json.loads(line)\n                except ValueError:continue\n                row=by_key.get((record.get(\'split\'),record.get(\'filename\')))\n                if row and reusable(record,row,config_hash) and not read_checkpoint(checkpoint_root,row,config_hash):\n                    atomic_json(checkpoint_path(checkpoint_root,row),record);restored+=1\n    return restored\n\n\ndef result_record(row,result,config,attempts):\n    text=result.text\n    if not isinstance(text,str) or not text.strip():raise ValueError(\'Model returned empty transcript\')\n    if result.mode!=\'verbatim\':raise ValueError(\'Unexpected model output mode\')\n    words=[asdict(w) for w in (result.words or [])]\n    warnings=0\n    for word in words:\n        start,end=word.get(\'start\'),word.get(\'end\')\n        if start is None or end is None or not math.isfinite(start) or not math.isfinite(end):\n            raise ValueError(\'Nonfinite word timestamp\')\n        if end<start or start<-.25 or end>row[\'duration_s\']+.5:warnings+=1\n    record={**row,\'transcript\':text,\'error\':None,\'model_id\':config[\'model_id\'],\n        \'model_revision\':config[\'model_revision\'],\'crisperwhisper_version\':config[\'package_version\'],\n        \'mode\':\'verbatim\',\'language\':result.language,\'backend\':config[\'backend\'],\n        \'compute_type\':config[\'compute_type\'],\'config_fingerprint\':fingerprint(config),\n        \'processing_time_s\':float(result.processing_time),\'attempts\':attempts,\n        \'transcribed_at_utc\':datetime.now(timezone.utc).isoformat(),\n        \'words\':words,\'timing_warning_count\':warnings,\n        \'chunks\':[asdict(c) for c in (result.chunks or [])]}\n    # Keep original model text; no spelling, grammar, filler or punctuation cleanup.\n    json.dumps(record,allow_nan=False)\n    return record\n\n\ndef worker(job_file,config_file,model_dir,checkpoint_root):\n    import torch\n    from crisperwhisper import CrisperWhisperModel\n    from importlib.metadata import version\n    jobs=json.loads(Path(job_file).read_text());config=json.loads(Path(config_file).read_text())\n    assert version(\'crisperwhisper\')==config[\'package_version\']\n    assert torch.cuda.is_available() and torch.cuda.device_count()==1\n    torch.set_num_threads(2);torch.manual_seed(config[\'seed\']);torch.cuda.manual_seed_all(config[\'seed\'])\n    signature=fingerprint(config)\n    remaining=[r for r in jobs if not read_checkpoint(checkpoint_root,r,signature)]\n    print(f\'{os.environ.get("WORKER_NAME","GPU")}: {len(jobs)-len(remaining)} cached, {len(remaining)} remaining\',flush=True)\n    if not remaining:return\n    model=CrisperWhisperModel(model_dir,backend=config[\'backend\'],device=\'cuda\',\n        device_index=0,compute_type=config[\'compute_type\'])\n    started=time.monotonic();processed_audio=0.;total_audio=sum(r[\'duration_s\'] for r in remaining)\n    for index,row in enumerate(remaining,1):\n        error=None;record=None\n        for attempt in range(1,config[\'max_attempts\']+1):\n            try:\n                result=model.transcribe(row[\'audio_path\'],**config[\'transcribe\'])\n                record=result_record(row,result,config,attempt);break\n            except Exception as exc:\n                error=f\'{type(exc).__name__}: {str(exc)[:1500]}\'\n                gc.collect();torch.cuda.empty_cache()\n                print(f\'{row["split"]}/{row["filename"]}: retry {attempt}/{config["max_attempts"]}\',flush=True)\n        if record is None:\n            record={**row,\'transcript\':None,\'error\':error,\'words\':[],\'chunks\':[],\n                \'model_id\':config[\'model_id\'],\'model_revision\':config[\'model_revision\'],\n                \'config_fingerprint\':signature,\'attempts\':config[\'max_attempts\']}\n        atomic_json(checkpoint_path(checkpoint_root,row),record)\n        processed_audio+=row[\'duration_s\'];elapsed=time.monotonic()-started\n        speed=processed_audio/max(elapsed,.001);eta=(total_audio-processed_audio)/max(speed,.001)\n        print(f\'{os.environ.get("WORKER_NAME","GPU")}: {index}/{len(remaining)} | \'\n            f\'{speed:.2f}x audio | ETA {eta/60:.1f} min | {row["split"]}/{row["filename"]} | \'\n            f\'{"OK" if record.get("error") is None else "FAILED"}\',flush=True)\n\n\ndef run_workers(jobs,config,model_dir,checkpoint_root,runtime_file,work_root,gpu_count):\n    root=Path(work_root);root.mkdir(parents=True,exist_ok=True)\n    partitions=[[] for _ in range(gpu_count)];loads=[0.]*gpu_count\n    signature=fingerprint(config)\n    pending=[r for r in jobs if not read_checkpoint(checkpoint_root,r,signature)]\n    for row in sorted(pending,key=lambda r:r[\'duration_s\'],reverse=True):\n        gpu=min(range(gpu_count),key=lambda i:loads[i]);partitions[gpu].append(row);loads[gpu]+=row[\'duration_s\']\n    config_file=root/\'config.json\';atomic_json(config_file,config)\n    processes=[]\n    try:\n        for gpu,partition in enumerate(partitions):\n            if not partition:continue\n            job_file=root/f\'jobs_gpu_{gpu}.json\';atomic_json(job_file,partition)\n            environment={k:v for k,v in os.environ.items() if k not in [\'KAGGLE_KEY\',\'KAGGLE_API_TOKEN\']}\n            environment.update(CUDA_VISIBLE_DEVICES=str(gpu),WORKER_NAME=f\'GPU {gpu}\',\n                OMP_NUM_THREADS=\'2\',TOKENIZERS_PARALLELISM=\'false\',HF_HUB_OFFLINE=\'1\')\n            command=[sys.executable,\'-u\',str(runtime_file),\'worker\',str(job_file),str(config_file),\n                str(model_dir),str(checkpoint_root)]\n            processes.append(subprocess.Popen(command,env=environment))\n        while any(p.poll() is None for p in processes):\n            failures=[p.returncode for p in processes if p.poll() is not None and p.returncode!=0]\n            if failures:raise RuntimeError(f\'GPU worker failed with exit code {failures}; checkpoints preserved\')\n            time.sleep(1)\n        if any(p.returncode!=0 for p in processes):raise RuntimeError(\'GPU worker failed; checkpoints preserved\')\n    finally:\n        for p in processes:\n            if p.poll() is None:p.terminate()\n        for p in processes:\n            try:p.wait(timeout=10)\n            except subprocess.TimeoutExpired:p.kill();p.wait()\n\n\ndef export_bundle(jobs,config,checkpoint_root,destination,data_root,sources,model_license=None):\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    records=[];word_rows=[];signature=fingerprint(config)\n    for row in jobs:\n        record=read_checkpoint(checkpoint_root,row,signature)\n        if record is None:\n            path=checkpoint_path(checkpoint_root,row)\n            try:previous=json.loads(path.read_text())\n            except (OSError,ValueError):previous={}\n            record={**row,\'transcript\':None,\'error\':previous.get(\'error\') or \'Missing/invalid checkpoint\',\n                \'words\':[],\'chunks\':[],\'model_id\':config[\'model_id\'],\n                \'model_revision\':config[\'model_revision\'],\'config_fingerprint\':signature}\n        else:\n            # Refresh ordering and current mounted path on a cross-session resume.\n            record={**record,**row}\n        records.append(record)\n        for index,word in enumerate(record.get(\'words\',[])):\n            word_rows.append(dict(split=row[\'split\'],filename=row[\'filename\'],audio_id=row[\'audio_id\'],\n                word_index=index,word=word[\'word\'],start_s=word[\'start\'],end_s=word[\'end\']))\n    with (destination/\'transcripts_crisper.jsonl\').open(\'w\') as handle:\n        for record in records:handle.write(json.dumps(record,ensure_ascii=False,allow_nan=False)+\'\\n\')\n    flat=[{**{k:v for k,v in r.items() if k not in [\'words\',\'chunks\']},\n        \'words_json\':json.dumps(r.get(\'words\',[]),ensure_ascii=False),\n        \'chunks_json\':json.dumps(r.get(\'chunks\',[]),ensure_ascii=False)} for r in records]\n    frame=pd.DataFrame(flat);frame[\'label\']=pd.array(frame.label,dtype=\'Float64\')\n    frame.to_csv(destination/\'transcripts_crisper.csv\',index=False)\n    frame.to_parquet(destination/\'transcripts_crisper.parquet\',index=False)\n    for split in [\'train\',\'test\']:\n        frame[frame.split.eq(split)].to_csv(destination/f\'{split}_transcripts_crisper.csv\',index=False)\n        shutil.copy2(Path(data_root)/f\'{split}.csv\',destination/f\'source_{split}.csv\')\n    words=pd.DataFrame(word_rows,columns=[\'split\',\'filename\',\'audio_id\',\'word_index\',\'word\',\'start_s\',\'end_s\'])\n    words.to_parquet(destination/\'word_timestamps.parquet\',index=False)\n    words.to_csv(destination/\'word_timestamps.csv\',index=False)\n    failures=frame[frame.error.fillna(\'\').ne(\'\')|frame.transcript.fillna(\'\').str.strip().eq(\'\')]\n    failures[[\'split\',\'filename\',\'error\']].to_csv(destination/\'transcription_errors.csv\',index=False)\n    schema={\'primary_key\':[\'split\',\'filename\'],\'row_order\':\'train then test, preserving source CSV order\',\n        \'label\':\'Original 0–5 training score; null for test. Test CSV labels are placeholders.\',\n        \'label_known\':\'True only for original training labels.\',\n        \'transcript\':\'Unmodified CrisperWhisper verbatim text, not human ground truth.\',\n        \'audio_relative_path\':\'Path relative to competition Dataset_Final directory.\',\n        \'words_json\':\'JSON list of {word,start,end}; seconds in original recording.\',\n        \'chunks_json\':\'Raw model chunk metadata; model may use overlapping windows.\',\n        \'error\':\'Null on success; incomplete exports must not be used as completed datasets.\',\n        \'timing_warning_count\':\'Number of unusual timestamp intervals; text is preserved.\',\n        \'all_columns\':{c:str(frame[c].dtype) for c in frame.columns}}\n    atomic_json(destination/\'schema.json\',schema)\n    from importlib.metadata import version,PackageNotFoundError\n    packages={}\n    for package in [\'crisperwhisper\',\'torch\',\'transformers\',\'numpy\',\'pandas\',\'soundfile\',\'pyarrow\',\'kaggle\']:\n        try:packages[package]=version(package)\n        except PackageNotFoundError:pass\n    manifest=dict(schema_version=1,created_at_utc=datetime.now(timezone.utc).isoformat(),\n        competition=\'shl-hiring-assessment-2026\',config=config,config_fingerprint=signature,\n        total_rows=len(frame),split_counts=frame.split.value_counts().to_dict(),errors=len(failures),\n        complete=len(failures)==0,source_csvs=sources,packages=packages,\n        label_policy=\'Train labels preserved. Test labels are unknown/null; source_test.csv retains placeholders.\',\n        transcript_policy=\'Verbatim model output; no grammar correction or filler removal.\',\n        model_license_url=f\'https://huggingface.co/{config["model_id"]}/blob/{config["model_revision"]}/LICENSE.md\')\n    atomic_json(destination/\'run_manifest.json\',manifest)\n    if model_license and Path(model_license).exists():shutil.copy2(model_license,destination/\'MODEL_LICENSE.md\')\n    (destination/\'README.md\').write_text(\n        \'1. SHL 2026 CrisperWhisper verbatim transcripts: all train/test recordings.\\n\\n\'\n        \'2. Join on split + filename. Filename values overlap across train/test. \'\n        \'Original training labels are unchanged; combined/test transcript labels are blank. \'\n        \'source_test.csv contains original placeholders, not ground-truth test scores.\\n\\n\'\n        \'3. Preserve transcript text, filler tags, repetitions and original grammar. \'\n        \'These are ASR predictions and can contain omissions or hallucinations.\\n\\n\'\n        \'4. Word/chunk times are in seconds; review timing_warning_count. \'\n        \'See schema.json, run_manifest.json and file_checksums.json for provenance.\\n\\n\'\n        \'5. Attach this private dataset in future notebooks and read transcripts_crisper.parquet \'\n        \'or train_transcripts_crisper.csv/test_transcripts_crisper.csv. \'\n        \'Keep source-related recordings together in scoring validation.\\n\\n\'\n        \'6. Competition data terms apply. CrisperWhisper 2.0 weights and generated outputs \'\n        \'carry the upstream non-commercial research licence; see MODEL_LICENSE.md. \'\n        \'This metadata does not grant additional rights.\\n\\n\'\n        \'7. To resume in a new session, attach this export containing transcripts_crisper.jsonl; \'\n        \'only matching successful model/config/audio checkpoints are reused.\\n\')\n    atomic_json(destination/\'file_checksums.json\',{p.name:digest(p) for p in sorted(destination.iterdir())\n        if p.is_file() and p.name not in [\'file_checksums.json\',\'dataset-metadata.json\']})\n    validate_bundle(destination,jobs,config)\n    return frame,manifest\n\n\ndef validate_bundle(destination,jobs,config,require_complete=False):\n    destination=Path(destination);frame=pd.read_parquet(destination/\'transcripts_crisper.parquet\')\n    if len(frame)!=len(jobs) or frame.duplicated([\'split\',\'filename\']).any():raise ValueError(\'Invalid export keys/counts\')\n    expected=pd.DataFrame(jobs)\n    if list(zip(frame.split,frame.filename))!=list(zip(expected.split,expected.filename)):\n        raise ValueError(\'Source CSV ordering changed\')\n    train=frame[frame.split.eq(\'train\')]\n    expected_train=expected[expected.split.eq(\'train\')]\n    if not np.array_equal(train.label.to_numpy(dtype=float),expected_train.label.to_numpy(dtype=float)):\n        raise ValueError(\'Training labels changed\')\n    test=frame[frame.split.eq(\'test\')]\n    if not test.label.isna().all() or test.label_known.any():raise ValueError(\'Test labels must be unknown\')\n    if not train.label_known.all():raise ValueError(\'Missing label-known flag\')\n    if not frame.model_id.eq(config[\'model_id\']).all() or not frame.model_revision.eq(config[\'model_revision\']).all():\n        raise ValueError(\'Export contains another model\')\n    manifest=json.loads((destination/\'run_manifest.json\').read_text())\n    if manifest[\'config_fingerprint\']!=fingerprint(config):raise ValueError(\'Configuration mismatch\')\n    for name,sha in json.loads((destination/\'file_checksums.json\').read_text()).items():\n        if Path(name).name!=name or digest(destination/name)!=sha:raise ValueError(\'Export checksum mismatch\')\n    failed=frame.error.fillna(\'\').ne(\'\')|frame.transcript.fillna(\'\').str.strip().eq(\'\')\n    if manifest[\'complete\']!=bool(not failed.any()) or manifest[\'errors\']!=int(failed.sum()):\n        raise ValueError(\'Manifest completion mismatch\')\n    if require_complete and failed.any():raise ValueError(\'Incomplete transcripts; rerun workers before publication\')\n    return frame\n\n\n\nif __name__ == "__main__":\n    if len(sys.argv)>1 and sys.argv[1]=="worker": worker(*sys.argv[2:])\n'

if RUN_TRANSCRIPTION:
    import importlib.util
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    RUNTIME_FILE = WORK_ROOT / "crisper_runtime.py"
    RUNTIME_FILE.write_text(RUNTIME_SOURCE)
    spec = importlib.util.spec_from_file_location("crisper_runtime", RUNTIME_FILE)
    runtime = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(runtime)

## 3. Read every recording and preserve labels

1. CSV and WAV identifiers must agree exactly within each split.
2. Audio checksums, duration, channel count and source row order are recorded.
3. Test CSV placeholder labels never become training targets.

In [ ]:
if RUN_TRANSCRIPTION:
    import json
    import pandas as pd
    from IPython.display import display
    INPUT_ROOT = Path("/kaggle/input")
    if DATA_ROOT_OVERRIDE:
        DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    else:
        candidates = [p.parent for p in INPUT_ROOT.rglob("train.csv")
                      if "shl-hiring-assessment-2026" in str(p)
                      and (p.parent / "test.csv").is_file()
                      and (p.parent / "train").is_dir()
                      and (p.parent / "test").is_dir()]
        assert len(candidates) == 1, "Attach the SHL Hiring Assessment 2026 competition, or set DATA_ROOT_OVERRIDE."
        DATA_ROOT = candidates[0]
    jobs, SOURCE_METADATA = runtime.make_jobs(DATA_ROOT, EXPECTED_COUNTS)
    CONFIG = {
        "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
        "package_version": CRISPER_VERSION, "backend": BACKEND,
        "compute_type": COMPUTE_TYPE, "seed": SEED, "max_attempts": MAX_ATTEMPTS,
        "transcribe": {"language": "en", "mode": "verbatim", "word_timestamps": WORD_TIMESTAMPS,
            "hallucination_mitigation": HALLUCINATION_MITIGATION,
            "longform_strategy": "continuation", "chunk_duration": 30.0,
            "temperature_fallback": True, "max_new_tokens": 256},
    }
    CONFIG_HASH = runtime.fingerprint(CONFIG)
    CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"
    restored = runtime.restore_previous(INPUT_ROOT, CHECKPOINT_ROOT, jobs, CONFIG_HASH)
    cached = sum(runtime.read_checkpoint(CHECKPOINT_ROOT, row, CONFIG_HASH) is not None for row in jobs)
    print(f"985 recordings | {sum(r['duration_s'] for r in jobs)/3600:.2f} audio hours | {cached} cached ({restored} restored)")
    overview = pd.DataFrame(jobs)
    display(overview.groupby("split").agg(recordings=("filename", "size"), audio_minutes=("duration_s", lambda x:x.sum()/60)))
    display(overview.loc[overview.split.eq("train"), "label"].value_counts().sort_index())

## 4. Download the pinned CrisperWhisper model

1. The matching tokenizer and audio processor are downloaded with the weights.
2. Model weights stay in the temporary cache; transcript dataset uploads contain
   transcripts, provenance and the licence, without model weights or audio.

In [ ]:
if RUN_TRANSCRIPTION:
    from huggingface_hub import snapshot_download
    patterns = ["*.json", "*.txt", "LICENSE.md", "README.md"]
    if cached < len(jobs):
        patterns += ["*.safetensors"]
    MODEL_DIR = Path(snapshot_download(
        repo_id=MODEL_ID, revision=MODEL_REVISION,
        cache_dir=str(MODEL_CACHE), allow_patterns=patterns,
    ))
    model_config = json.loads((MODEL_DIR / "config.json").read_text())
    assert model_config["model_type"] == "whisper"
    assert (MODEL_DIR / "tokenizer.json").exists()
    assert (MODEL_DIR / "preprocessor_config.json").exists()
    assert (MODEL_DIR / "LICENSE.md").exists()
    if cached < len(jobs):
        assert (MODEL_DIR / "model.safetensors").exists()
    print("Pinned model ready:", MODEL_ID, MODEL_REVISION)
        if cached < len(jobs):
            import subprocess, sys, shutil
            # Fresh process prevents stale imports from an upstream CT2 installation.
            converter_source = "from crisperwhisper.converter import ensure_ct2_model\nimport json, sys\np=ensure_ct2_model(sys.argv[1], quantization='float16', cache_dir=sys.argv[2])\nprint(json.dumps({'path':str(p)}))\n"
            print("Converting the pinned model to CT2 once; this setup stage uses CPU.")
            conversion = subprocess.run(
                [sys.executable, "-c", converter_source, str(MODEL_DIR), str(MODEL_CACHE / "converted")],
                capture_output=True, text=True,
            )
            if conversion.returncode:
                print(conversion.stderr[-5000:])
                raise RuntimeError("CT2 conversion failed. Original notebook/checkpoints remain available.")
            converted_dir = Path(json.loads(conversion.stdout.strip().splitlines()[-1])["path"])
            assert (converted_dir / "model.bin").exists()
            shutil.copy2(MODEL_DIR / "LICENSE.md", converted_dir / "LICENSE.md")
            MODEL_DIR = converted_dir
            print("Optimized CT2 model ready for both GPU workers:", MODEL_DIR)


## 5. Transcribe on Kaggle GPUs

1. Progress and ETA appear below. Every finished recording is checkpointed.
2. Failed recordings retry individually; rerunning this cell retries failures
   and retains successful transcripts.
3. An interruption writes a partial export in `crisper_dataset` for recovery.
   The final validation requires a complete successful export.

In [ ]:
if RUN_TRANSCRIPTION:
    try:
        runtime.run_workers(jobs, CONFIG, MODEL_DIR, CHECKPOINT_ROOT,
                            RUNTIME_FILE, WORK_ROOT, GPU_COUNT)
    finally:
        transcripts, manifest = runtime.export_bundle(
            jobs, CONFIG, CHECKPOINT_ROOT, OUTPUT_DIR, DATA_ROOT, SOURCE_METADATA,
            model_license=MODEL_DIR / "LICENSE.md",
        )
        print(f"Exported {len(transcripts)} rows; failures: {manifest['errors']}; complete: {manifest['complete']}")
        print("Export folder:", OUTPUT_DIR)
    runtime.validate_bundle(OUTPUT_DIR, jobs, CONFIG, require_complete=True)
    print("All 985 recordings validated. Train labels preserved; test labels null.")

## 6. Inspect and download the output

1. `transcripts_crisper.csv` and `.parquet`: every transcript and its metadata.
2. `train_transcripts_crisper.csv` / `test_transcripts_crisper.csv`: reusable split files.
3. `word_timestamps.csv` / `.parquet`: words with their original-audio timing.
4. `transcripts_crisper.jsonl`: full model results and portable resume checkpoints.
5. `schema.json`, `run_manifest.json`, checksums, source CSVs and model licence.

In [ ]:
if RUN_TRANSCRIPTION:
    from IPython.display import Audio, FileLink, display
    display(transcripts[["split", "filename", "label", "label_known", "transcript", "error"]].head(10))
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file(): display(FileLink(str(path)))
    example = transcripts.loc[transcripts.split.eq("train") & transcripts.label.eq(0)].iloc[0]
    print(f"Zero-labelled example: {example.filename} | transcript: {example.transcript}")
    display(Audio(example.audio_path))

## 7. Save reusable outputs on Kaggle

1. After validation, save a notebook version with its output files.
2. Attach the saved notebook outputs as inputs in future scoring notebooks,
   or create a private dataset from the export files through Kaggle's website.
3. Dataset publication is separate from inference and needs no credentials
   in this notebook. All audio transcription runs on Kaggle GPUs.

In [ ]:
if RUN_TRANSCRIPTION:
    runtime.validate_bundle(OUTPUT_DIR, jobs, CONFIG, require_complete=True)
    print("All 985 transcripts are saved in", OUTPUT_DIR)
    print("Save this notebook version with outputs to retain them on Kaggle.")

## 8. Use these transcripts in a later scoring notebook

1. Add your completed private dataset as an input to the scoring notebook.
2. Read the split files directly or merge using split/filename; avoid merging
   the combined file by filename alone.

```python
from pathlib import Path
import pandas as pd

root = next(Path('/kaggle/input').rglob('train_transcripts_crisper.csv')).parent
train = pd.read_csv(root / 'train_transcripts_crisper.csv')
test = pd.read_csv(root / 'test_transcripts_crisper.csv')
assert train.label.notna().all()
assert test.label.isna().all()
# train['transcript'], train['label'] are paired in original source CSV order.
```